In [ ]:
import os
import json
import pandas as pd
import concurrent.futures
import aanalytics2 as api2
from datetime import datetime
import difflib
import copy
from dotenv import load_dotenv


In [ ]:
load_dotenv()

In [ ]:
def dataInitiatorS():
    api2.importConfigFile(os.path.join(auth.auth, 'aanalyticsact_auth.json'))
    logger = api2.Login()
    logger.connector.config

dataInitiatorS()
ags = api2.Analytics("YOUR_REPORT_SUITE")

In [ ]:
def initialize_api():
    current_dir = os.path.dirname(os.path.abspath(__file__))
    config_path = os.path.join(current_dir, 'aanalyticsact_auth.json')
    
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"❌ 인증 파일을 찾을 수 없습니다: {config_path}")
        
    # 1. API에 JSON 설정 파일 임포트
    api2.importConfigFile(config_path)
    api2.Login()
    
    # 2. JSON 파일에서 REPORT_SUITE_ID(또는 필요한 ID)를 동적으로 읽어오기
    with open(config_path, 'r', encoding='utf-8') as f:
        config_data = json.load(f)
    
    # JSON 내부에 저장된 키 이름에 맞게 가져옵니다 (예: 'REPORT_SUITE_ID' 또는 'rsid')
    rsid = config_data.get("REPORT_SUITE_ID") 
    if not rsid:
        # 만약 JSON 안에 다른 이름으로 저장되어 있다면 그에 맞게 수정 가능합니다.
        rsid = config_data.get("rsid", "YOUR_DEFAULT_REPORT_SUITE")
        
    return api2.Analytics(rsid)

In [ ]:


# 1. API 호출 시뮬레이션 (사용 중인 라이브러리에 맞게 수정)
def fetch_segment_data(segment_id):
    try:
        rsids = ags.getSegment(segment_id, True)

        # print(f"Fetching: {segment_id}...")
        return {"id": segment_id, 
                "name": rsids['name'], 
                "definition": rsids['definition'],
                "description": rsids.get('description', ""), 
                "full_form" : rsids} # 가짜 데이터
    except Exception as e:
        print(f"Error fetching {segment_id}: {e}")
        return None

# 2. 메인 엔진 클래스
class SegmentManager:
    def __init__(self):
        self.all_segments = {}      # {id: json_content}
        self.name_to_id = {}        # {name: id}
        self.dependency_map = {}    # {parent_name: [child_names]}

    def load_segments_parallel(self, id_list, max_workers=10):
        """멀티스레딩으로 세그먼트 데이터를 빠르게 로드"""
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            results = list(executor.map(fetch_segment_data, id_list))
        
        for data in results:
            if data:
                self.all_segments[data['id']] = data
                self.name_to_id[data['name']] = data['id']
                
        print("✅ 모든 세그먼트 다운로드 완료!")
        self._build_maps()

    def _build_maps(self):
        """로딩된 데이터를 바탕으로 의존성 트리 구축"""
        all_names = set(self.name_to_id.keys())

        for seg_id, content in self.all_segments.items():
            parent_name = content.get("name")
            children = self._extract_descriptions(content.get("definition", {}), all_names, parent_name)
            if children:
                self.dependency_map[parent_name] = list(set(children))

    def _extract_descriptions(self, obj, valid_names, current_name):
        """재귀적으로 definition 내부의 자식 세그먼트 이름 추출"""
        found = []
        if isinstance(obj, dict):
            desc = obj.get("description")
            if desc in valid_names and desc != current_name:
                found.append(desc)
            for v in obj.values():
                found.extend(self._extract_descriptions(v, valid_names, current_name))
        elif isinstance(obj, list):
            for item in obj:
                found.extend(self._extract_descriptions(item, valid_names, current_name))
        return found

    def get_affected_parents(self, child_name):
        """특정 자식을 포함하고 있는 부모 리스트 확인 (영향도 분석)"""
        return [p for p, children in self.dependency_map.items() if child_name in children]

    def get_all_descriptions_per_segment(segment_manager):
        """
        기존 SegmentManager 객체를 입력받아, 
        각 세그먼트별로 내부 JSON(definition)에 숨어있는 
        모든 'description' 타이틀을 추출하여 딕셔너리로 반환합니다.
        """
        def _recursive_extract(obj):
            """내부 JSON 구조를 재귀적으로 돌며 description 값을 추출하는 내부 함수"""
            found = []
            if isinstance(obj, dict):
                if "description" in obj and obj["description"]:
                    found.append(obj["description"])
                for v in obj.values():
                    found.extend(_recursive_extract(v))
            elif isinstance(obj, list):
                for item in obj:
                    found.extend(_recursive_extract(item))
            return found
    
        # 결과를 담을 딕셔너리 { 세그먼트_ID: [description_리스트] }
        result_map = {}
    
        # manager 내부의 모든 세그먼트 데이터를 순회
        for seg_id, content in segment_manager.all_segments.items():
            definition_json = content.get("definition", {})
            
            # 재귀 함수 호출 후 중복 제거
            all_titles = _recursive_extract(definition_json)
            result_map[seg_id] = list(set(all_titles))
    
        return result_map
# def print_all_relationships(self):
#     """모든 부모-자식 관계를 계층적으로 출력"""
#     print("\n" + "="*50)
#     print("      [전체 세그먼트 의존성 리포트]      ")
#     print("="*50)

#     if not self.dependency_map:
#         print("연결된 부모-자식 관계가 없습니다.")
#         return

#     for parent, children in self.dependency_map.items():
#         print(f"\n📂 부모: {parent}")
#         for child in children:
#             # 해당 자식의 ID도 함께 표시 (name_to_id 맵 활용)
#             child_id = self.name_to_id.get(child, "Unknown ID")
#             print(f"   ┗━━ 🧩 자식: {child} ({child_id})")

#     # 통계: 가장 많이 쓰이는 '인기 부품' 찾기
#     self.print_usage_statistics()

# def print_usage_statistics(self):
#     """어떤 자식 세그먼트가 가장 많은 부모에게 포함되어 있는지 계산"""
#     usage_count = {}
#     for children in self.dependency_map.values():
#         for child in children:
#             usage_count[child] = usage_count.get(child, 0) + 1
    
#     # 많이 사용된 순으로 정렬
#     sorted_usage = sorted(usage_count.items(), key=lambda x: x[1], reverse=True)

#     print("\n" + "-"*50)
#     print("📊 자식 세그먼트별 사용 빈도 (영향도 Top 5)")
#     print("-"*50)
#     for name, count in sorted_usage[:5]:
#         print(f"- {name}: {count}개의 부모 세그먼트에서 사용 중")
# print(manager.dependency_map)

def show_me_everything(manager):
    print("\n[전체 세그먼트 의존성 계층도]")
    for parent, children in manager.dependency_map.items():
        print(f"\n📂 {parent} (ID: {manager.name_to_id.get(parent)})")
        for child in children:
            print(f"   └── 🧩 {child}")
            
def build_reverse_impact_map(manager):
    # 역방향 지도를 담을 딕셔너리 { 자식명: [부모명1, 부모명2, ...] }
    reverse_map = {}

    # dependency_map은 { 부모: [자식리스트] } 구조임
    for parent_name, children_list in manager.dependency_map.items():
        for child_name in children_list:
            if child_name not in reverse_map:
                reverse_map[child_name] = []
            
            # 이 자식의 부모 리스트에 현재 부모 추가
            reverse_map[child_name].append(parent_name)
            
    return reverse_map

def check_manager_stats(manager):
    total_segments = len(manager.all_segments)
    names_indexed = len(manager.name_to_id)
    parent_count = len(manager.dependency_map)
    
    # 중복을 제외한 모든 자식 세그먼트의 수 계산
    all_children = set()
    for children in manager.dependency_map.values():
        all_children.update(children)
    
    print(f"📊 [Manager 데이터 적재 현황]")
    print(f"- 로드된 전체 세그먼트 수: {total_segments}개")
    print(f"- 이름 색인(Indexing) 완료: {names_indexed}개")
    print(f"- 관계가 발견된 부모 세그먼트: {parent_count}개")
    print(f"- 부품으로 사용된 자식 세그먼트: {len(all_children)}개")
    print("-" * 30)

def calculate_similarity(obj1, obj2):
    """두 객체 간의 유사도를 퍼센트(%)로 계산"""
    str1 = json.dumps(obj1, sort_keys=True)
    str2 = json.dumps(obj2, sort_keys=True)
    
    # SequenceMatcher를 이용한 문자열 유사도 측정
    matcher = difflib.SequenceMatcher(None, str1, str2)
    return round(matcher.ratio() * 100, 2)
def check_integrity_with_score(manager, target_child_name):
    child_id = manager.name_to_id.get(target_child_name)
    if not child_id:
        print(f"❌ '{target_child_name}'을 찾을 수 없습니다.")
        return []

    # 최신 로직 (기준점)
    latest_pred = manager.all_segments[child_id]["definition"]["container"]["pred"]
    affected_parents = manager.get_affected_parents(target_child_name)
    
    analysis_result = []

    for p_name in affected_parents:
        p_id = manager.name_to_id.get(p_name)
        parent_json = manager.all_segments[p_id]

        # 내부 pred 추출 (기존 로직 유지)
        def find_internal_pred(obj, target):
            if isinstance(obj, dict):
                if obj.get("description") == target and "pred" in obj:
                    return obj["pred"]
                for v in obj.values():
                    res = find_internal_pred(v, target)
                    if res: return res
            elif isinstance(obj, list):
                for item in obj:
                    res = find_internal_pred(item, target)
                    if res: return res
            return None

        current_internal_pred = find_internal_pred(parent_json, target_child_name)

        if current_internal_pred:
            # 유사도 계산
            similarity = calculate_similarity(latest_pred, current_internal_pred)
            
            # 불일치할 경우만 리스트에 추가 (유사도 100% 미만)
            if similarity < 100:
                analysis_result.append({
                    "name": p_name,
                    "id": p_id,
                    "similarity": similarity,
                    "diff_score": round(100 - similarity, 2) # 불일치 정도
                })

    # 결과 출력: 불일치 정도(diff_score)가 높은 순으로 정렬 (가장 심각한 것부터)
    analysis_result = sorted(analysis_result, key=lambda x: x['diff_score'], reverse=True)

    print(f"\n🎯 대상 자식: {target_child_name} ({child_id})")
    print(f"📊 분석 결과: 총 {len(affected_parents)}개의 부모 중 {len(analysis_result)}개에서 불일치(수정 필요) 발견")
    print("="*80)
    print(f"{'No.':<4} | {'Similarity':<12} | {'Parent Name':<30} | {'ID'}")
    print("-" * 80)

    if not analysis_result:
        print("✅ 모든 부모 세그먼트의 로직이 100% 일치합니다.")
    else:
        for i, p in enumerate(analysis_result, 1):
            color_warn = "⚠️" if p['similarity'] < 50 else "🔔" # 50% 미만이면 매우 위험
            print(f"{i:02d}  | {p['similarity']:>8}% {color_warn} | {p['name']:<30} | {p['id']}")
    
    print("="*80)
    return analysis_result



def report_all_outdated_segments(manager):
    """
    모든 자식 세그먼트를 순회하며, 불일치가 발견된 모든 부모를 리포팅합니다.
    """
    
    def calculate_similarity(obj1, obj2):
        str1 = json.dumps(obj1, sort_keys=True)
        str2 = json.dumps(obj2, sort_keys=True)
        return round(difflib.SequenceMatcher(None, str1, str2).ratio() * 100, 1)

    def extract_pred(obj, target):
        if isinstance(obj, dict):
            if obj.get("description") == target and "pred" in obj:
                return obj["pred"]
            for v in obj.values():
                res = extract_pred(v, target)
                if res: return res
        elif isinstance(obj, list):
            for item in obj:
                res = extract_pred(item, target)
                if res: return res
        return None

    # 1. 모든 자식 이름 수집 (Dependency Map 기준)
    # 만약 특정 자식만 나온다면, dependency_map에 다른 자식들이 들어있는지 확인이 필요합니다.
    all_child_names = set()
    for children_list in manager.dependency_map.values():
        all_child_names.update(children_list)

    final_report = {}

    # 2. 루프: 모든 자식을 하나씩 검사
    for child_name in sorted(list(all_child_names)): # 자식 이름순 정렬
        child_id = manager.name_to_id.get(child_name)
        if not child_id:
            continue
        
        # 자식의 최신 기준 로직
        latest_pred = manager.all_segments[child_id]["definition"]["container"]["pred"]
        
        # 이 자식을 포함하는 모든 부모 탐색
        affected_parents = manager.get_affected_parents(child_name)
        outdated_list = []

        for p_name in affected_parents:
            p_id = manager.name_to_id.get(p_name)
            parent_json = manager.all_segments[p_id]
            current_pred_in_parent = extract_pred(parent_json, child_name)

            if current_pred_in_parent:
                sim = calculate_similarity(latest_pred, current_pred_in_parent)
                if sim < 100: # 불일치(100% 미만)인 경우만 수집
                    outdated_list.append({
                        "name": p_name,
                        "id": p_id,
                        "similarity": sim
                    })

        # 불일치한 부모가 1개라도 있는 자식만 리포트에 추가
        if outdated_list:
            final_report[child_name] = {
                "id": child_id,
                "parents": sorted(outdated_list, key=lambda x: x['name'])
            }

    # 3. 결과 출력부
    print("\n" + "빨간색" if not final_report else "파란색") # (개념적 구분)
    print("="*90)
    print(f"📊 [종합 점검] 총 {len(all_child_names)}개의 자식 중 {len(final_report)}개에서 불일치 감지")
    print("="*90)

    if not final_report:
        print("✅ 모든 세그먼트의 동기화 상태가 완벽합니다 (100% 일치).")
        return {}

    for child_name, data in final_report.items():
        print(f"\n[자식] {child_name} ({data['id']})")
        print(f"ㄴ ⚠️ 불일치 부모 수: {len(data['parents'])}개")
        print(f"   {'No.':<4} | {'Match':<8} | {'Child Name':<30} | {'Parent Name':<30} | {'Parent ID'}")
        print(f"   {'-' * 85}")
        
        for i, p in enumerate(data['parents'], 1):
            icon = "🔴" if p['similarity'] < 60 else "🟡"
            print(f"   {i:02d}  | {p['similarity']:>5}% {icon} | {child_name:<30} | {p['name']:<30} | {p['id']}")
        
    print("\n" + "="*90)
    return final_report

##################################################################################
##################################################################################
##################################################################################
##################################################################################

import json
import os
import copy
from datetime import datetime


def sync_and_push_with_report(ags, manager, target_child_name, comment=None):
    today_str = datetime.now().strftime("%Y-%m-%d")
    today_str_description = datetime.now().strftime("%y%m%d")
    backup_dir = f"backup_{datetime.now().strftime('%Y%m%d')}"
    if not os.path.exists(backup_dir): os.makedirs(backup_dir)

    detailed_reports = []
    
    # [STEP 1] 자식 정보 로드 (변경 없음)
    child_id = manager.name_to_id.get(target_child_name)
    if not child_id: return []
    latest_pred = manager.all_segments[child_id]["definition"]["container"]["pred"]
    
    # [STEP 2] 불일치 부모 탐지
    outdated_parents_info = check_integrity_with_score(manager, target_child_name)
    if not outdated_parents_info: return []

    for p_info in outdated_parents_info:
        seg_id = p_info['id']
        seg_name = p_info['name']
        
        # 3-1. 백업: 'full_form'에 담긴 서버 응답 원본을 보관
        original_full_json = copy.deepcopy(manager.all_segments[seg_id]['full_form'])
        backup_path = f"{backup_dir}/FULL_BACKUP_{seg_id}.json"
        
        try:
            with open(backup_path, 'w', encoding='utf-8') as f:
                json.dump(original_full_json, f, ensure_ascii=False, indent=4)
        except Exception as e:
            print(f"⚠️ 백업 실패: {e}"); continue

        # 3-2. 업데이트 작업: 원본(full_form)을 복사해서 수정 시작
        updated_full_json = copy.deepcopy(original_full_json)
        
        # [A] 로직 교체: updated_full_json 내부의 'definition'을 직접 수정
        def replace_logic(obj):
            changed = False
            if isinstance(obj, dict):
                if obj.get("description") == target_child_name and "pred" in obj:
                    obj["pred"] = copy.deepcopy(latest_pred)
                    return True
                for v in obj.values():
                    if replace_logic(v): changed = True
            elif isinstance(obj, list):
                for item in obj:
                    if replace_logic(item): changed = True
            return changed

        replace_logic(updated_full_json.get("definition", {}))
        
        # [B] Description(커멘트) 업데이트: full_form의 최상위 description 필드를 수정
        if comment:
            new_entry = f"{today_str_description} - {comment}"
            # full_form 내부에 직접 접근하여 기존 설명을 가져옵니다.
            current_desc = updated_full_json.get("description", "")
            if current_desc:
                updated_full_json["description"] = f"{current_desc}\n{new_entry}"
            else:
                updated_full_json["description"] = new_entry

        # 3-3. 서버 전송: 수정된 full_form을 그대로 보냄
        try:
            ags.updateSegment(seg_id, updated_full_json)
            status = "Success"
            
            # 중요: 서버 성공 시 manager의 데이터도 full_form과 하위 필드들을 모두 최신화
            manager.all_segments[seg_id]['full_form'] = updated_full_json
            manager.all_segments[seg_id]['description'] = updated_full_json.get("description", "")
            manager.all_segments[seg_id]['definition'] = updated_full_json.get("definition")
            
            print(f"✅ 반영 완료: {seg_name}")
        except Exception as e:
            status = f"Failed ({str(e)})"
            print(f"❌ 반영 실패: {seg_name}")

        detailed_reports.append({
            "날짜": today_str, "세그먼트명": seg_name, "세그먼트ID": seg_id,
            "요인세그먼트명": target_child_name, "추가커멘트": comment or "", "결과": status
        })

    return detailed_reports
    
def sync_multiple_to_single_excel(ags, manager, target_child_names, comments):
    """
    여러 자식 세그먼트와 그에 대응하는 커멘트 리스트를 받아 처리 후 하나의 엑셀로 저장
    """
    combined_reports = []
    start_time = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    
    # target_child_names와 comments의 길이를 맞춤 (부족하면 빈 문자열)
    if len(comments) < len(target_child_names):
        comments += [""] * (len(target_child_names) - len(comments))

    print(f"🚀 총 {len(target_child_names)}개의 세그먼트 통합 업데이트 시작 (Description 업데이트 포함)...")

    for name, comment in zip(target_child_names, comments):
        print(f"\n[대상: {name} | 커멘트: {comment}]")
        report = sync_and_push_with_report(ags, manager, name, comment)
        combined_reports.extend(report)

    if combined_reports:
        df = pd.DataFrame(combined_reports)
        count_str = f"{len(target_child_names)}건"
        file_name = f"sync_report_total_{count_str}_{start_time}.xlsx"
        
        try:
            # 기존 스타일링 함수(save_report_to_excel) 로직 활용 권장
            df.to_excel(file_name, index=False)
            print(f"\n✨ [최종 완료] 리포트 저장됨: {file_name}")
        except Exception as e:
            print(f"❌ 엑셀 저장 실패: {e}")
    else:
        print("⚠️ 작업 데이터가 없습니다.")

    return combined_reports

    

In [ ]:
if __name__ == "__main__":
    
    manager = SegmentManager()
    # 10개의 스레드를 동시에 돌려 광속 호출!
    manager.load_segments_parallel(seglist, max_workers=10)

    # 예시: 특정 세그먼트가 수정되었을 때 영향받는 부모 찾기
    target = "[Global] Cart Page Visit"
    parents = manager.get_affected_parents(target)
    print(f"\n📢 '{target}' 수정 시 업데이트가 필요한 부모 리스트:")
    print(parents)

In [ ]:
len(parents)

In [ ]:
if __name__ == "__main__":
    # 1. API 로그인
    print("📡 Adobe Analytics API 연결 중...")
    ags = initialize_api()
    
    # 2. 분석할 CSV 파일 경로 (이 경로는 본인 PC 상황에 맞게 적어주세요)
    csv_path = "C:/Path/Segment ID List.csv"
    
    if os.path.exists(csv_path):
        df_target = pd.read_csv(csv_path)
        seg_id_list = df_target['ColumnName'].dropna().tolist()
        
        # 3. 매니저 구동 및 데이터 적재
        manager = SegmentManager()
        manager.load_segments_parallel(ags, seg_id_list, max_workers=10)
    else:
        print(f"⚠️ 지정된 경로에 CSV 파일이 없습니다: {csv_path}")

In [ ]:
print(len(manager.get_affected_parents("Segment Name")))

In [ ]:
show_me_everything(manager)

In [ ]:

full_impact_map = build_reverse_impact_map(manager)

for child, parents in full_impact_map.items():
    print(f"🧩 자식: {child}")
    print(f"   └── 🏠 포함된 부모들: {parents}")

In [ ]:

check_manager_stats(manager)

In [ ]:

target = "Segment Name"
results = check_integrity_with_score(manager, target)

In [ ]:
full_report = report_all_outdated_segments(manager)

In [ ]:
#############################################################
#############################################################
#############################################################
#############################################################
#############################################################
#############################################################
childSegment =["Segment Name1","Segment Name2"]
comments = ["Comment1","Comment2"]

reports = sync_multiple_to_single_excel(ags, manager, childSegment,comments)

# # 2. 리포트가 존재할 경우 엑셀 저장
# if reports:
#     save_report_to_excel(reports,"sync_report"+len(childSegment))